# Paper 4 — Edge/cloud CA-IEDI measurement

Measures deterministic UTF-8 wire bytes and keeps estimated transfer time separate from observed gateway round-trip time. The default loopback is a protocol dry run, not a 4G experiment.


In [ ]:
from pathlib import Path
import os
import sys

search_roots = (Path.cwd(), *Path.cwd().parents, Path("/content/iedi-mas"))
ROOT = next((path for path in search_roots if (path / "src" / "iedi").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Repository not found. Clone it and install with: pip install -e .[gemini]")
sys.path.insert(0, str(ROOT / "src"))

from iedi.codebook import Codebook
from iedi.providers import GoogleGenAIProvider, OfflineFixtureProvider
from iedi.pipeline import build_pipeline
from iedi.schemas import InterpretationRequest

codebook = Codebook.from_json(ROOT / "data" / "codebook.demo.json")
# OfflineFixtureProvider only echoes reviewed evidence; it is never empirical evidence.
# Set IEDI_LIVE_GEMINI=1 and GEMINI_API_KEY to exercise the real 2.5 Flash/Pro adapter.
LIVE_GEMINI = os.getenv("IEDI_LIVE_GEMINI") == "1"
provider = GoogleGenAIProvider() if LIVE_GEMINI else OfflineFixtureProvider()
print("provider:", "live Gemini" if LIVE_GEMINI else "offline schema fixture")

pipeline = build_pipeline("paper4", codebook=codebook, provider=provider, config_path=ROOT / "configs" / "paper4.json")


In [ ]:
from iedi.edge import NetworkProfile
from iedi.transport import (
    CloudInterpretationService, EdgeInterpretationClient,
    LoopbackEdgeCloudTransport,
)

network = NetworkProfile(
    name="documented-4g-emulation",
    uplink_kbps=10_000,
    downlink_kbps=20_000,
    base_rtt_ms=45,
    emulator="replace with actual emulator/hardware metadata",
)
# Loopback exercises the exact protocol. Replace it with
# HttpEdgeCloudTransport for a separately deployed cloud service.
cloud = CloudInterpretationService(pipeline)
edge = EdgeInterpretationClient(LoopbackEdgeCloudTransport(cloud))
observation = edge.interpret(
    InterpretationRequest("An unresolved phrase"),
    raw_audio_bytes=96 * 1024,  # hypothetical baseline, not a measured file
    network_profile=network,
)
{
    "route": observation.result["decision"]["used_route"],
    "wire_bytes": observation.wire_payload_bytes,
    "reduction": observation.payload_reduction,
    "estimated_transfer_ms": observation.estimated_transfer_ms,
    "observed_gateway_rtt_ms": observation.observed_gateway_round_trip_ms,
}


Run repeated trials and retain raw records before comparing with the paper. Target byte/latency values are not embedded as successful assertions.
